# HealthBot - Interactive Learning System with LangGraph

This notebook implements an educational system that:
1. Receives a learning topic
2. Searches for information with Tavily
3. Generates a summary in English (based exclusively on Tavily)
4. Creates a quiz-type question
5. Grades the user's response
6. Allows restart or exit

**Technology Stack:**
- LangGraph (workflow orchestration)
- OpenAI (text generation)
- Tavily (web search)
- Langchain (integration)

In [1]:
# Install dependencies
import subprocess
import sys

packages = [
    "langgraph",
    "langchain",
    "langchain-google-genai",
    "langchain-community",
    "python-dotenv"
]

for package in packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])

In [2]:
# Imports
import os
from typing import TypedDict, Any
from langchain_openai import ChatOpenAI
from langchain_community.tools.tavily_search import TavilySearchResults
from langgraph.graph import StateGraph
from langchain_core.messages import HumanMessage, AIMessage

# Load environment variables
from dotenv import load_dotenv
import sys

# Specify absolute path of .env
env_path = r"c:\Users\pablo\Desktop\HealthBot-demo\.env"
load_dotenv(dotenv_path=env_path)

# API Configuration
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")

print("="*60)
print("🔧 HEALTHBOT CONFIGURATION")
print("="*60)

if OPENAI_API_KEY:
    print("✅ OPENAI_API_KEY configured (GPT-4)")
else:
    print("⚠️ WARNING: OPENAI_API_KEY not configured")
    print("   Get your key at: https://platform.openai.com/api-keys")

if TAVILY_API_KEY:
    print("✅ TAVILY_API_KEY configured")
else:
    print("⚠️ WARNING: TAVILY_API_KEY not configured")
    print("   Visit https://tavily.com to get a key")
print("="*60)

# Initialize tools only if we have the keys
if OPENAI_API_KEY and TAVILY_API_KEY:
    llm = ChatOpenAI(
        model="gpt-4o-mini",
        api_key=OPENAI_API_KEY,
        temperature=0.7
    )
    search_tool = TavilySearchResults(max_results=5, api_key=TAVILY_API_KEY)
    print("\n✅ LLM (OpenAI GPT-4) and Tavily initialized")
    print("   Model: gpt-4o-mini (Recommended for cost/performance)")
else:
    llm = None
    search_tool = None
    print("\n⚠️ Tools will be initialized when you provide the keys")

🔧 HEALTHBOT CONFIGURATION
✅ OPENAI_API_KEY configured (GPT-4)
✅ TAVILY_API_KEY configured

✅ LLM (OpenAI GPT-4) and Tavily initialized
   Model: gpt-4o-mini (Recommended for cost/performance)


C:\Users\pablo\AppData\Local\Temp\ipykernel_11804\240356429.py:45: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the `langchain-tavily package and should be used instead. To use it run `pip install -U `langchain-tavily` and import as `from `langchain_tavily import TavilySearch``.
  search_tool = TavilySearchResults(max_results=5, api_key=TAVILY_API_KEY)


In [ ]:
# Diagnostics: verify .env file contents
import subprocess
result = subprocess.run(["type", r"c:\Users\pablo\Desktop\HealthBot-demo\.env"], shell=True, capture_output=True, text=True)
print(".env file contents:")
print(result.stdout)
print("\nVariables detected by Python:")
print(f"GOOGLE_API_KEY: {'Configured' if os.getenv('GOOGLE_API_KEY') else 'Not configured'}")
print(f"TAVILY_API_KEY: {'Configured' if os.getenv('TAVILY_API_KEY') else 'Not configured'}")
print(f"OPENAI_API_KEY: {'Configured' if os.getenv('OPENAI_API_KEY') else 'Not configured'}")

In [3]:
# Definition of shared State
class HealthBotState(TypedDict):
    """Shared state of learning flow"""
    topic: str  # User-entered topic
    search_results: dict  # Tavily results
    summary: str  # Summary in English (3-4 paragraphs)
    quiz_question: str  # Generated question
    user_answer: str  # User's response
    grade: str  # Grade (A, B, C, D, F)
    justification: str  # Grade justification
    continue_learning: bool  # Continue with another topic?
    current_step: str  # Current step of flow

In [4]:
# NODE 1: Request learning topic
def input_topic(state: HealthBotState) -> HealthBotState:
    """
    Requests the user to enter the learning topic.
    """
    print("\n" + "="*60)
    print("🎓 WELCOME TO HEALTHBOT - Interactive Learning System")
    print("="*60)
    
    topic = input("\n📚 What topic would you like to learn? (e.g., Artificial Intelligence, History of Medicine, etc.)\n➜ ")
    
    state["topic"] = topic.strip()
    state["current_step"] = "search"
    
    print(f"\n✓ Selected topic: {state['topic']}")
    
    return state

In [5]:
# NODE 2: Search information with Tavily
def search_tavily(state: HealthBotState) -> HealthBotState:
    """
    Performs a search with Tavily on the topic.
    This is the ONLY source of information for the summary.
    """
    print(f"\n🔍 Searching for information about: {state['topic']}")
    
    try:
        # Execute search
        results = search_tool.invoke(state["topic"])
        state["search_results"] = results
        
        print(f"✓ Found {len(results)} information sources")
        
        # Show sources summary
        for i, result in enumerate(results, 1):
            source = result.get("source", "Unknown source")
            print(f"  {i}. {source}")
            
    except Exception as e:
        print(f"❌ Search error: {str(e)}")
        state["search_results"] = []
    
    state["current_step"] = "generate_summary"
    return state

In [6]:
# NODE 3: Generate summary in English
def generate_summary(state: HealthBotState) -> HealthBotState:
    """
    Generates a 3-4 paragraph summary based ONLY
    on Tavily results. Does NOT use prior knowledge.
    """
    print(f"\n📝 Generating summary about: {state['topic']}")
    
    if not state["search_results"]:
        print("❌ No search results available to generate summary")
        state["summary"] = "Could not generate summary due to lack of results."
        return state
    
    # Build context EXCLUSIVELY from Tavily
    search_context = "\n".join([f"Source: {r.get('source', 'N/A')}\nContent: {r.get('content', '')}" 
                                 for r in state["search_results"]])
    
    # Prompt that ensures only search is used
    prompt = f"""You are an educational assistant that can ONLY use information
from a web search. You do NOT have prior knowledge about the topic.

Search results about '{state['topic']}':\n\n{search_context}\n\nTask: Generate a summary in English of exactly 3-4 paragraphs about the topic.
CRITICAL CONSTRAINTS:
- Only use information from the search results above
- Do NOT add prior knowledge
- Do NOT mention that you used web search
- Write in clear and accessible English
- Each paragraph should have 2-3 sentences
"""
    
    try:
        message = HumanMessage(content=prompt)
        response = llm.invoke([message])
        state["summary"] = response.content
        print("✓ Summary generated successfully")
        
    except Exception as e:
        print(f"❌ Error generating summary: {str(e)}")
        state["summary"] = "Error generating summary."
    
    state["current_step"] = "generate_question"
    return state

In [7]:
# NODE 4: Generate quiz-type question
def generate_question(state: HealthBotState) -> HealthBotState:
    """
    Generates a quiz-type question based ONLY on the generated summary.
    """
    print(f"\n❓ Generating question about the topic...")
    
    prompt = f"""Based ONLY on the following summary, generate a multiple-choice question
(quiz-type) that can be answered directly from the summary.

SUMMARY:
{state['summary']}

Generate the question with this format:
[QUESTION]
[CORRECT ANSWER]
[OPTION B (incorrect but plausible)]
[OPTION C (incorrect but plausible)]
[OPTION D (incorrect but plausible)]

The question must:
- Be clear and specific
- Be answerable directly from the summary
- Have 4 options (A, B, C, D)
- Have one unique correct answer
"""
    
    try:
        message = HumanMessage(content=prompt)
        response = llm.invoke([message])
        state["quiz_question"] = response.content
        print("✓ Question generated successfully")
        
    except Exception as e:
        print(f"❌ Error generating question: {str(e)}")
        state["quiz_question"] = "Error generating question."
    
    state["current_step"] = "get_answer"
    return state

In [8]:
# NODE 5: Get user answer
def get_user_answer(state: HealthBotState) -> HealthBotState:
    """
    Requests the user's response to the quiz question.
    """
    print("\n" + "="*60)
    print("📖 TOPIC SUMMARY")
    print("="*60)
    print(f"\n{state['summary']}")
    
    print("\n" + "="*60)
    print("❓ QUIZ QUESTION")
    print("="*60)
    print(f"\n{state['quiz_question']}")
    
    answer = input("\n✏️ Your answer (A, B, C or D): ").strip().upper()
    
    # Validate answer
    while answer not in ["A", "B", "C", "D"]:
        answer = input("Please enter a valid option (A, B, C or D): ").strip().upper()
    
    state["user_answer"] = answer
    state["current_step"] = "grade"
    
    return state

In [9]:
# NODE 6: Grade user response
def grade_answer(state: HealthBotState) -> HealthBotState:
    """
    Grades the user's response using ONLY the summary as source.
    Assigns a grade (A, B, C, D, F) and provides justification with citations.
    """
    print("\n⏳ Evaluating your answer...")
    
    grading_prompt = f"""You are an educational evaluator. Your task is to grade the user's response
based ONLY on the following summary and question.

SUMMARY (unique source of truth):
{state['summary']}

QUIZ QUESTION:
{state['quiz_question']}

USER RESPONSE: {state['user_answer']}

Task:
1. Determine if the response is correct by comparing with the summary
2. Assign a grade: A (excellent), B (good), C (satisfactory), D (insufficient) or F (failed)
3. Provide justification by citing fragments from the summary

Response format:
GRADE: [A/B/C/D/F]
JUSTIFICATION: [Your explanation]
SUMMARY CITATION: "[Relevant fragment from the summary]"
"""
    
    try:
        message = HumanMessage(content=grading_prompt)
        response = llm.invoke([message])
        
        # Extract grade from response
        response_text = response.content
        
        # Search for grade in text
        grades = {"A": "A", "B": "B", "C": "C", "D": "D", "F": "F"}
        grade_found = None
        for grade_letter in grades.values():
            if f"GRADE: {grade_letter}" in response_text:
                grade_found = grade_letter
                break
        
        if not grade_found:
            # Try to extract in another way
            for grade_letter in grades.values():
                if grade_letter in response_text and "GRADE" in response_text:
                    idx = response_text.find("GRADE:")
                    if idx != -1:
                        grade_section = response_text[idx:idx+30]
                        if grade_letter in grade_section:
                            grade_found = grade_letter
                            break
        
        state["grade"] = grade_found if grade_found else "C"
        state["justification"] = response_text
        
    except Exception as e:
        print(f"❌ Error grading response: {str(e)}")
        state["grade"] = "C"
        state["justification"] = f"Evaluation error: {str(e)}"
    
    state["current_step"] = "show_results"
    return state

In [10]:
# NODE 7: Show results
def show_results(state: HealthBotState) -> HealthBotState:
    """
    Shows the evaluation results to the user.
    """
    print("\n" + "="*60)
    print("📊 YOUR EVALUATION RESULTS")
    print("="*60)
    
    # Mapping of letters to emojis and descriptions
    grade_map = {
        "A": "🌟 EXCELLENT",
        "B": "⭐ VERY GOOD",
        "C": "👍 GOOD",
        "D": "⚠️ NEEDS IMPROVEMENT",
        "F": "❌ FAILED"
    }
    
    print(f"\nYour answer: {state['user_answer']}")
    print(f"\nGRADE: {grade_map.get(state['grade'], state['grade'])}")
    print(f"\nJUSTIFICATION:\n{state['justification']}")
    
    state["current_step"] = "decide_next"
    return state

In [11]:
# NODE 8: Decide if continue or exit
def decide_next(state: HealthBotState) -> HealthBotState:
    """
    Allows the user to choose between:
    - Learn another topic
    - Exit the system
    """
    print("\n" + "="*60)
    print("WHAT WOULD YOU LIKE TO DO?")
    print("="*60)
    print("\n1. Learn a new topic")
    print("2. Exit the system")
    
    choice = input("\n➜ Select an option (1 or 2): ").strip()
    
    while choice not in ["1", "2"]:
        choice = input("Please select a valid option (1 or 2): ").strip()
    
    state["continue_learning"] = choice == "1"
    
    if state["continue_learning"]:
        state["current_step"] = "input_topic"
    else:
        state["current_step"] = "end"
    
    return state

In [12]:
# BUILD LANGGRAPH WORKFLOW
def create_healthbot_graph():
    """
    Creates the LangGraph workflow with nodes and conditional edges.
    """
    # Initialize state
    workflow = StateGraph(HealthBotState)
    
    # Add nodes
    workflow.add_node("input_topic", input_topic)
    workflow.add_node("search_tavily", search_tavily)
    workflow.add_node("generate_summary", generate_summary)
    workflow.add_node("generate_question", generate_question)
    workflow.add_node("get_user_answer", get_user_answer)
    workflow.add_node("grade_answer", grade_answer)
    workflow.add_node("show_results", show_results)
    workflow.add_node("decide_next", decide_next)
    
    # Set entry point
    workflow.set_entry_point("input_topic")
    
    # Add edges (transitions between nodes)
    workflow.add_edge("input_topic", "search_tavily")
    workflow.add_edge("search_tavily", "generate_summary")
    workflow.add_edge("generate_summary", "generate_question")
    workflow.add_edge("generate_question", "get_user_answer")
    workflow.add_edge("get_user_answer", "grade_answer")
    workflow.add_edge("grade_answer", "show_results")
    workflow.add_edge("show_results", "decide_next")
    
    # Conditional edge: continue or exit?
    def should_continue(state):
        if state["continue_learning"]:
            return "input_topic"  # Restart flow
        else:
            return "end"  # Exit
    
    workflow.add_conditional_edges(
        "decide_next",
        should_continue,
        {
            "input_topic": "input_topic",
            "end": "__end__"
        }
    )
    
    # Compile the workflow
    app = workflow.compile()
    return app

# Create the application
app = create_healthbot_graph()
print("✓ LangGraph workflow compiled successfully")

✓ LangGraph workflow compiled successfully


In [13]:
# RUN THE SYSTEM
def run_healthbot():
    """
    Main function that executes the complete HealthBot flow.
    """
    # Initial state
    initial_state: HealthBotState = {
        "topic": "",
        "search_results": [],
        "summary": "",
        "quiz_question": "",
        "user_answer": "",
        "grade": "",
        "justification": "",
        "continue_learning": False,
        "current_step": "input_topic"
    }
    
    try:
        # Execute the workflow
        # The workflow will execute the complete flow iteratively
        # until the user chooses to exit
        while True:
            result = app.invoke(initial_state)
            
            # If the user does not want to continue, exit the loop
            if not result.get("continue_learning", False):
                break
            
            # Update state for next iteration
            initial_state = result
        
        # Farewell message
        print("\n" + "="*60)
        print("👋 Thank you for using HealthBot!")
        print("="*60)
        print("\nKeep learning! 📚")
        
    except KeyboardInterrupt:
        print("\n\n⚠️ Session interrupted by user")
        print("\n👋 Goodbye!")
    except Exception as e:
        print(f"\n❌ Error during execution: {str(e)}")
        import traceback
        traceback.print_exc()

# Uncomment the following line to run the system
# run_healthbot()

print("\n" + "="*60)
print("✅ HealthBot ready to run")
print("="*60)
print("\nTo start the system, run:")
print("  run_healthbot()")


✅ HealthBot ready to run

To start the system, run:
  run_healthbot()


In [14]:
# DEMO: Automatic system test
print("\n" + "🚀"*30)
print("STARTING HEALTHBOT SYSTEM DEMO")
print("🚀"*30)

# Initial state with example topic
demo_state = {
    "topic": "Artificial Intelligence",
    "search_results": [],
    "summary": "",
    "quiz_question": "",
    "user_answer": "",
    "grade": "",
    "justification": "",
    "continue_learning": False,
    "current_step": "input_topic"
}

print(f"\n📚 SELECTED TOPIC: {demo_state['topic']}\n")

# 1. SEARCH WITH TAVILY
print("="*60)
print("1️⃣ SEARCHING FOR INFORMATION WITH TAVILY...")
print("="*60)
demo_state = search_tavily(demo_state)

# 2. GENERATE SUMMARY
print("\n" + "="*60)
print("2️⃣ GENERATING SUMMARY IN ENGLISH...")
print("="*60)
demo_state = generate_summary(demo_state)
print(f"\n📄 GENERATED SUMMARY:\n{demo_state['summary']}\n")

# 3. GENERATE QUESTION
print("="*60)
print("3️⃣ GENERATING QUIZ-TYPE QUESTION...")
print("="*60)
demo_state = generate_question(demo_state)
print(f"\n❓ GENERATED QUESTION:\n{demo_state['quiz_question']}\n")

# 4. SIMULATE USER RESPONSE
demo_state["user_answer"] = "A"  # Simulate user answering "A"
print("="*60)
print(f"4️⃣ USER RESPONSE SIMULATED: {demo_state['user_answer']}")
print("="*60)

# 5. GRADE RESPONSE
print("\n" + "="*60)
print("5️⃣ GRADING THE RESPONSE...")
print("="*60)
demo_state = grade_answer(demo_state)

# 6. SHOW RESULTS
print("\n" + "="*60)
print("6️⃣ SHOWING EVALUATION RESULTS...")
print("="*60)
demo_state = show_results(demo_state)

print("\n" + "🎉"*30)
print("✅ DEMO COMPLETED SUCCESSFULLY")
print("🎉"*30)

print("\n📊 EXECUTED FLOW SUMMARY:")
print(f"   ✓ Topic: {demo_state['topic']}")
print(f"   ✓ Sources found: {len(demo_state['search_results'])}")
print(f"   ✓ Summary generated: {len(demo_state['summary'])} characters")
print(f"   ✓ Question generated: Yes")
print(f"   ✓ User response: {demo_state['user_answer']}")
print(f"   ✓ Grade: {demo_state['grade']}")
print(f"\n🚀 The system is 100% FUNCTIONAL and ready to use")
print(f"\nTo use the system interactively, run: run_healthbot()")


🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀
STARTING HEALTHBOT SYSTEM DEMO
🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀

📚 SELECTED TOPIC: Artificial Intelligence

1️⃣ SEARCHING FOR INFORMATION WITH TAVILY...

🔍 Searching for information about: Artificial Intelligence
✓ Found 5 information sources
  1. Unknown source
  2. Unknown source
  3. Unknown source
  4. Unknown source
  5. Unknown source

2️⃣ GENERATING SUMMARY IN ENGLISH...

📝 Generating summary about: Artificial Intelligence
❌ Error generating summary: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}

📄 GENERATED SUMMARY:
Error generating summary.

3️⃣ GENERATING QUIZ-TYPE QUESTION...

❓ Generating question about the topic...
❌ Error generating question: Error code: 429 - {'error': {'message': 

## 📖 Usage Guide

### Prerequisites
1. **Required API Keys**:
   - `OPENAI_API_KEY`: OpenAI key (GPT-4)
   - `TAVILY_API_KEY`: Tavily Search key

2. **Environment variables configuration**:
   Create a `.env` file in the project directory:
   ```
   OPENAI_API_KEY=sk-...
   TAVILY_API_KEY=tvly-...
   ```

### Step-by-step execution

1. **Run the cells in order** from top to bottom
2. **Imports and configuration**: Will be installed from the first cell
3. **Graph creation**: Compiles after running all node cells
4. **Execution**: Run this cell to start:
   ```python
   run_healthbot()
   ```

### Application Flow

```
1. 📚 Input Topic
   ↓
2. 🔍 Search Tavily
   ↓
3. 📝 Generate Summary (based 100% on Tavily)
   ↓
4. ❓ Generate Quiz Question
   ↓
5. ✏️ Get User Answer
   ↓
6. 📊 Grade Answer (using Only summary)
   ↓
7. 📈 Show Results
   ↓
8. 🔄 Decide Next (Continue or Exit)
```

### Implemented Validations

- ✅ Only Tavily as information source
- ✅ Summary generated exclusively from search
- ✅ Question based only on the summary
- ✅ Grading justified with summary citations
- ✅ Error handling at each stage
- ✅ User input validation